# Phase 5: Expirementation with Kronecker Sequences

First we import

In [1]:
import math
import itertools

Then we create a more efficeint algorithm for rho computation, for kronecker sequences

In [2]:
# ============================================================
# Helpers
# ============================================================

def centered_mod(x, N):
    """
    Return the centered representative of x mod N in (-N/2, N/2].
    """
    r = x % N
    if r > N / 2:
        r -= N
    return int(r)

def alpha_to_g_vector(N, alpha):
    """
    Convert alpha = (g2/N, ..., gd/N) back to integer generator g = (1, g2, ..., gd).
    """
    return (1,) + tuple(int(round(a * N)) % N for a in alpha)

def singleton_upper_bound(N, g_tail):
    """
    Initial exact upper bound from vectors with one tail coordinate = ±1, others 0.
    For such vectors, tail product = 1, and h1 is the centered residue of ±gj mod N.
    """
    best_rho = float("inf")
    best_h = None
    m = len(g_tail)

    for j, gj in enumerate(g_tail):
        for sign in (+1, -1):
            tail = [0] * m
            tail[j] = sign
            h1 = centered_mod(-(sign * gj), N)
            rho = max(1, abs(h1))  # tail product is 1
            if rho < best_rho:
                best_rho = rho
                best_h = (h1,) + tuple(tail)

    return int(best_rho), best_h

def short_relation_rho_one(N, g_tail):
    """
    Cheap exact pre-check for rho = 1.
    We only need to test tail coordinates in {-1,0,1}, not all zero.
    If the centered h1 is also in {-1,0,1}, then rho = 1.
    """
    m = len(g_tail)

    for tail in itertools.product([-1, 0, 1], repeat=m):
        if all(x == 0 for x in tail):
            continue

        s = sum(g * h for g, h in zip(g_tail, tail))
        h1 = centered_mod(-s, N)

        if abs(h1) <= 1:
            return True, (h1,) + tail

    return False, None


# ============================================================
# Generic fast exact solver for rank-1 generators
# ============================================================

def rho_rank1_fast_exact(N, g_vector):
    """
    Exact Zaremba index for rank-1 generator g = (1, g2, ..., gd).

    Uses:
    - short-relation rho=1 precheck
    - singleton upper bound
    - branch-and-bound recursion on tail coordinates
    - symmetry: first nonzero tail coordinate is forced positive

    Returns
    -------
    best_rho : int
    best_h   : tuple
    """
    if g_vector[0] != 1:
        raise ValueError("Expected generator in the form (1, g2, ..., gd).")

    g_tail = tuple(int(x) for x in g_vector[1:])
    m = len(g_tail)  # tail dimension

    # 1. Fast exact check for rho = 1
    found_one, h_one = short_relation_rho_one(N, g_tail)
    if found_one:
        return 1, h_one

    # 2. Initial upper bound from singleton tails
    best_rho, best_h = singleton_upper_bound(N, g_tail)

    # 3. Recursive exact search
    current = [0] * m

    def recurse(idx, partial_prod, sum_mod, started):
        nonlocal best_rho, best_h

        # If even current partial product already loses, prune
        if partial_prod >= best_rho:
            return

        if idx == m:
            if not started:
                return  # tail all zero -> invalid h = 0
            h1 = centered_mod(-sum_mod, N)
            score = partial_prod * max(1, abs(h1))
            if score < best_rho:
                best_rho = int(score)
                best_h = (h1,) + tuple(current)
            return

        # Largest allowed |h_i| before factor alone kills the branch
        max_abs = (best_rho - 1) // partial_prod
        if max_abs < 0:
            return

        # Symmetry reduction:
        # before the first nonzero tail coordinate, only try h >= 0.
        if not started:
            candidate_values = range(0, int(max_abs) + 1)
        else:
            candidate_values = range(-int(max_abs), int(max_abs) + 1)

        gi = g_tail[idx]

        for h in candidate_values:
            # Enforce first nonzero tail coordinate positive
            if not started and h < 0:
                continue

            factor = max(1, abs(h))
            new_prod = partial_prod * factor
            if new_prod >= best_rho:
                continue

            current[idx] = h
            new_started = started or (h != 0)
            new_sum_mod = (sum_mod + gi * h) % N

            recurse(idx + 1, new_prod, new_sum_mod, new_started)

        current[idx] = 0

    recurse(idx=0, partial_prod=1, sum_mod=0, started=False)
    return int(best_rho), best_h


# ============================================================
# 4D / 5D wrappers
# ============================================================

def rho_4d_fast_exact_from_alpha(N, alpha):
    """
    Exact fast 4D rho from alpha = (g2/N, g3/N, g4/N).
    """
    g_vector = alpha_to_g_vector(N, alpha)
    if len(g_vector) != 4:
        raise ValueError("Expected 4D alpha with length 3.")
    return rho_rank1_fast_exact(N, g_vector)

def rho_5d_fast_exact_from_alpha(N, alpha):
    """
    Exact fast 5D rho from alpha = (g2/N, g3/N, g4/N, g5/N).
    """
    g_vector = alpha_to_g_vector(N, alpha)
    if len(g_vector) != 5:
        raise ValueError("Expected 5D alpha with length 4.")
    return rho_rank1_fast_exact(N, g_vector)

Example Usage:

4d: 
````
alpha = tuple(g / N for g in g_vector[1:])
rho_val, _ = rho_4d_fast_exact_from_alpha(N, alpha)
````